structured output

models can be requested to provide their response in a format match a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. langchain supports multiple schema types and methods for enforcing structured output


pydantic models:
for fields, description, nested structures

In [2]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:llama-3.3-70b-versatile")


In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="Movie title")
    year: int = Field(description="Release year")
    rating: float = Field(description="Movie rating (0-10)")
    director: str = Field(description="Director name")

In [5]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002D019CD6C10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002D019E411D0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Movie title', 'type': 'string'}, 'year': {'description': 'Release year', 'type': 'integer'}, 'rating': {'description': 'Movie rating (0-10)', 'type': 'number'}, 'director': {'description': 'Director name', 'type': 'string'}}, 'required': ['title', 'year', 'rating', 'director'], 'type': 'object'}}}

In [ ]:
response=model_with_structure.invoke('detail about saiyaara movie')


In [7]:
response

Movie(title='Saiyaara', year=2023, rating=6.5, director='Unknown')

In [13]:
class Actor(BaseModel):
    name:str
    role:str

class Movie1(BaseModel):
    title: str 
    year: int 
    rating: float 
    director: str
    cast:list[Actor]

message output along side parsed structure

In [14]:
model_with_structure1=model.with_structured_output(Movie1,include_raw=True)


In [15]:
response=model_with_structure1.invoke('detail about ready movie')
response


{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'yshtkr2mp', 'function': {'arguments': '{"cast":[{"name":"Salman Khan","role":"Prem Kapoor"},{"name":"Asin","role":"Pia S Malik"}],"director":"Anees Bazmee","rating":6.2,"title":"Ready","year":2011}', 'name': 'Movie1'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 280, 'total_tokens': 349, 'completion_time': 0.27659411, 'completion_tokens_details': None, 'prompt_time': 0.052790655, 'prompt_tokens_details': None, 'queue_time': 0.161833492, 'total_time': 0.329384765}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d6c13-c887-79a0-95ee-355cf05b2c16-0', tool_calls=[{'name': 'Movie1', 'args': {'cast': [{'name': 'Salman Khan', 'role': 'Prem Kapoor'}, {'name': 'Asin', 'role': 'Pia S Malik'}], 'director': 'Anees Bazmee'

TypedDict
idle when u don't need runtime validation 

In [16]:
from typing import TypedDict, Annotated
from typing_extensions import NotRequired

class Movie(TypedDict):
    title: Annotated[str, "Movie title (non-empty string)"]
    year: Annotated[int, "Release year between 1800 and 2100"]
    rating: Annotated[float, "Rating between 0 and 10"]
    director: NotRequired[Annotated[str, "Director name"]]

to see what all things model supports use profile 

In [17]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True}

In [18]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name:str
    email:str
    phone:str

agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{"role":"user","content":"extract contact info from : John Deo, john@example.com,98012-12111"}]
})
result

{'messages': [HumanMessage(content='extract contact info from : John Deo, john@example.com,98012-12111', additional_kwargs={}, response_metadata={}, id='15cd20d0-ef7b-413b-87ee-2a52f7c6ce4f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6wmse5s0v', 'function': {'arguments': '{"email":"john@example.com","name":"John Deo","phone":"98012-12111"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 260, 'total_tokens': 292, 'completion_time': 0.062624023, 'completion_tokens_details': None, 'prompt_time': 0.025130634, 'prompt_tokens_details': None, 'queue_time': 0.176893416, 'total_time': 0.087754657}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d6c35-1be5-7ef3-91f3-2f4cb18f2a3c-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john@ex

In [19]:
result["structured_response"]

ContactInfo(name='John Deo', email='john@example.com', phone='98012-12111')